# LLMOps: Monitoring and Observability

**Module 16: LLMOps & Production**  
**Objective**: Master production monitoring for LLM systems

Production LLM Monitoring:
- Logging and Tracing
- Metrics Collection
- Cost Tracking
- Performance Monitoring
- Error Detection
- User Feedback
- Debugging Production Issues

## What You'll Learn
1. LLM-specific monitoring
2. Distributed tracing
3. Cost tracking per request
4. Performance dashboards
5. Error detection and alerting
6. User feedback loops

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import time
import json
from collections import defaultdict, deque
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

## 1. LLM Observability Stack

**Key Monitoring Layers**:

```
┌─────────────────────────────────────┐
│       Business Metrics              │
│  (User satisfaction, conversion)    │
└─────────────────────────────────────┘
           ↑                           
┌─────────────────────────────────────┐
│       Application Metrics           │
│  (Latency, accuracy, cost)         │
└─────────────────────────────────────┘
           ↑                           
┌─────────────────────────────────────┐
│       Model Metrics                 │
│  (Tokens, quality, safety)         │
└─────────────────────────────────────┘
           ↑                           
┌─────────────────────────────────────┐
│       Infrastructure                │
│  (GPU, memory, throughput)         │
└─────────────────────────────────────┘
```

**Monitoring Tools**:
- **LangSmith**: LangChain tracing
- **Weights & Biases**: Experiment tracking
- **Prometheus + Grafana**: Infrastructure metrics
- **DataDog**: Full-stack observability
- **OpenTelemetry**: Distributed tracing

In [ ]:
@dataclass
class RequestMetrics:
    """Metrics for a single LLM request."""
    request_id: str
    timestamp: datetime
    user_id: str
    model: str
    
    # Input
    prompt: str
    prompt_tokens: int
    
    # Output
    completion: str
    completion_tokens: int
    
    # Performance
    latency_ms: float
    time_to_first_token_ms: float
    tokens_per_second: float
    
    # Cost
    cost_usd: float
    
    # Quality
    success: bool
    error_message: Optional[str] = None
    user_feedback: Optional[int] = None  # 1-5 rating
    
    # Safety
    flagged: bool = False
    moderation_scores: Dict[str, float] = field(default_factory=dict)

@dataclass
class Span:
    """Tracing span for distributed systems."""
    span_id: str
    trace_id: str
    parent_id: Optional[str]
    name: str
    start_time: datetime
    end_time: Optional[datetime]
    duration_ms: Optional[float]
    attributes: Dict = field(default_factory=dict)
    
class LLMLogger:
    """Production LLM logging system."""
    
    def __init__(self):
        self.requests: List[RequestMetrics] = []
        self.spans: Dict[str, List[Span]] = defaultdict(list)
        
    def log_request(self, request: RequestMetrics):
        """Log LLM request."""
        self.requests.append(request)
        
        # In production, send to logging backend
        # logger.info(json.dumps(dataclasses.asdict(request)))
        
    def start_span(self, trace_id: str, name: str, parent_id: Optional[str] = None) -> Span:
        """Start a new tracing span."""
        span = Span(
            span_id=f"span_{len(self.spans[trace_id])}",
            trace_id=trace_id,
            parent_id=parent_id,
            name=name,
            start_time=datetime.now(),
            end_time=None,
            duration_ms=None
        )
        self.spans[trace_id].append(span)
        return span
    
    def end_span(self, span: Span):
        """End a tracing span."""
        span.end_time = datetime.now()
        span.duration_ms = (span.end_time - span.start_time).total_seconds() * 1000
    
    def get_metrics_summary(self) -> Dict:
        """Calculate summary metrics."""
        if not self.requests:
            return {}
        
        latencies = [r.latency_ms for r in self.requests]
        costs = [r.cost_usd for r in self.requests]
        successes = sum(1 for r in self.requests if r.success)
        total_tokens = sum(r.prompt_tokens + r.completion_tokens for r in self.requests)
        
        return {
            "total_requests": len(self.requests),
            "success_rate": successes / len(self.requests) * 100,
            "avg_latency_ms": np.mean(latencies),
            "p50_latency_ms": np.percentile(latencies, 50),
            "p95_latency_ms": np.percentile(latencies, 95),
            "p99_latency_ms": np.percentile(latencies, 99),
            "total_cost_usd": sum(costs),
            "avg_cost_per_request_usd": np.mean(costs),
            "total_tokens": total_tokens,
            "flagged_requests": sum(1 for r in self.requests if r.flagged)
        }

# Example usage
logger = LLMLogger()

# Simulate some requests
print("Simulating LLM requests...\n")

for i in range(100):
    # Simulate request metrics
    prompt_tokens = random.randint(10, 500)
    completion_tokens = random.randint(50, 1000)
    latency = random.gauss(250, 50)  # ms
    ttft = random.gauss(50, 10)  # time to first token
    
    # Cost calculation (example: $0.002/1K tokens)
    cost = (prompt_tokens + completion_tokens) / 1000 * 0.002
    
    request = RequestMetrics(
        request_id=f"req_{i}",
        timestamp=datetime.now() - timedelta(seconds=100-i),
        user_id=f"user_{random.randint(1, 10)}",
        model="gpt-4",
        prompt=f"Example prompt {i}",
        prompt_tokens=prompt_tokens,
        completion=f"Example completion {i}",
        completion_tokens=completion_tokens,
        latency_ms=latency,
        time_to_first_token_ms=ttft,
        tokens_per_second=completion_tokens / (latency / 1000),
        cost_usd=cost,
        success=random.random() > 0.05,  # 95% success rate
        error_message=None if random.random() > 0.05 else "Timeout",
        user_feedback=random.randint(1, 5) if random.random() > 0.7 else None,
        flagged=random.random() < 0.02,  # 2% flagged
        moderation_scores={
            "hate": random.random() * 0.1,
            "violence": random.random() * 0.1,
            "sexual": random.random() * 0.1
        }
    )
    
    logger.log_request(request)

# Get summary
summary = logger.get_metrics_summary()

print("Metrics Summary:")
print("="*70)
print(f"Total Requests: {summary['total_requests']}")
print(f"Success Rate: {summary['success_rate']:.1f}%")
print(f"\nLatency:")
print(f"  Average: {summary['avg_latency_ms']:.1f} ms")
print(f"  P50: {summary['p50_latency_ms']:.1f} ms")
print(f"  P95: {summary['p95_latency_ms']:.1f} ms")
print(f"  P99: {summary['p99_latency_ms']:.1f} ms")
print(f"\nCost:")
print(f"  Total: ${summary['total_cost_usd']:.2f}")
print(f"  Per Request: ${summary['avg_cost_per_request_usd']:.4f}")
print(f"  Per 1M Tokens: ${summary['avg_cost_per_request_usd'] * 1000:.2f}")
print(f"\nSafety:")
print(f"  Flagged Requests: {summary['flagged_requests']}")
print(f"  Total Tokens: {summary['total_tokens']:,}")

In [ ]:
def visualize_monitoring_dashboard():
    """Visualize production monitoring dashboard."""
    
    fig = plt.figure(figsize=(18, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    # 1. Latency over time
    ax1 = fig.add_subplot(gs[0, :])
    ax1.set_title('Request Latency Over Time', fontsize=13, weight='bold')
    
    times = [r.timestamp for r in logger.requests]
    latencies = [r.latency_ms for r in logger.requests]
    
    ax1.plot(range(len(times)), latencies, 'b-', alpha=0.5, linewidth=1)
    # Moving average
    window = 10
    moving_avg = np.convolve(latencies, np.ones(window)/window, mode='valid')
    ax1.plot(range(window-1, len(times)), moving_avg, 'r-', linewidth=2,
            label=f'{window}-request MA')
    
    # P95 line
    p95 = summary['p95_latency_ms']
    ax1.axhline(p95, color='orange', linestyle='--', linewidth=2,
               label=f'P95: {p95:.1f}ms')
    
    ax1.set_xlabel('Request Number')
    ax1.set_ylabel('Latency (ms)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Latency distribution
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.set_title('Latency Distribution', fontsize=12, weight='bold')
    ax2.hist(latencies, bins=30, color='blue', alpha=0.7, edgecolor='black')
    ax2.axvline(summary['p50_latency_ms'], color='green', linestyle='--',
               linewidth=2, label='P50')
    ax2.axvline(summary['p95_latency_ms'], color='orange', linestyle='--',
               linewidth=2, label='P95')
    ax2.axvline(summary['p99_latency_ms'], color='red', linestyle='--',
               linewidth=2, label='P99')
    ax2.set_xlabel('Latency (ms)')
    ax2.set_ylabel('Frequency')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    # 3. Cost breakdown
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.set_title('Cost Breakdown', fontsize=12, weight='bold')
    
    # Group by user
    user_costs = defaultdict(float)
    for r in logger.requests:
        user_costs[r.user_id] += r.cost_usd
    
    users = list(user_costs.keys())[:8]  # Top 8
    costs = [user_costs[u] for u in users]
    
    ax3.bar(range(len(users)), costs, color='green', alpha=0.7, edgecolor='black')
    ax3.set_xlabel('User')
    ax3.set_ylabel('Cost ($)')
    ax3.set_xticks(range(len(users)))
    ax3.set_xticklabels([u.split('_')[1] for u in users], fontsize=9)
    ax3.grid(True, alpha=0.3, axis='y')
    
    # 4. Token usage
    ax4 = fig.add_subplot(gs[1, 2])
    ax4.set_title('Token Usage', fontsize=12, weight='bold')
    
    prompt_tokens_total = sum(r.prompt_tokens for r in logger.requests)
    completion_tokens_total = sum(r.completion_tokens for r in logger.requests)
    
    tokens = [prompt_tokens_total, completion_tokens_total]
    labels = ['Prompt', 'Completion']
    colors = ['lightblue', 'lightcoral']
    
    wedges, texts, autotexts = ax4.pie(tokens, labels=labels, colors=colors,
                                       autopct='%1.1f%%', startangle=90)
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_weight('bold')
        autotext.set_fontsize(11)
    
    # 5. Success rate
    ax5 = fig.add_subplot(gs[2, 0])
    ax5.set_title('Success vs Errors', fontsize=12, weight='bold')
    
    successes = sum(1 for r in logger.requests if r.success)
    failures = len(logger.requests) - successes
    
    ax5.bar(['Success', 'Error'], [successes, failures],
           color=['green', 'red'], alpha=0.7, edgecolor='black', linewidth=2)
    ax5.set_ylabel('Count')
    ax5.grid(True, alpha=0.3, axis='y')
    
    # Add percentage labels
    ax5.text(0, successes, f'{successes}\n({successes/len(logger.requests)*100:.1f}%)',
            ha='center', va='bottom', fontsize=11, weight='bold')
    ax5.text(1, failures, f'{failures}\n({failures/len(logger.requests)*100:.1f}%)',
            ha='center', va='bottom', fontsize=11, weight='bold')
    
    # 6. User feedback
    ax6 = fig.add_subplot(gs[2, 1])
    ax6.set_title('User Feedback Ratings', fontsize=12, weight='bold')
    
    ratings = [r.user_feedback for r in logger.requests if r.user_feedback]
    if ratings:
        rating_counts = [ratings.count(i) for i in range(1, 6)]
        colors_bar = ['red', 'orange', 'yellow', 'lightgreen', 'green']
        ax6.bar(range(1, 6), rating_counts, color=colors_bar,
               alpha=0.7, edgecolor='black', linewidth=2)
        ax6.set_xlabel('Rating')
        ax6.set_ylabel('Count')
        ax6.set_xticks(range(1, 6))
        ax6.grid(True, alpha=0.3, axis='y')
        
        avg_rating = np.mean(ratings)
        ax6.text(0.98, 0.95, f'Avg: {avg_rating:.2f}⭐',
                transform=ax6.transAxes, ha='right', va='top',
                fontsize=11, weight='bold',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # 7. Throughput
    ax7 = fig.add_subplot(gs[2, 2])
    ax7.set_title('Tokens per Second', fontsize=12, weight='bold')
    
    tps_values = [r.tokens_per_second for r in logger.requests]
    ax7.hist(tps_values, bins=25, color='purple', alpha=0.7, edgecolor='black')
    ax7.axvline(np.mean(tps_values), color='red', linestyle='--',
               linewidth=2, label=f'Avg: {np.mean(tps_values):.0f}')
    ax7.set_xlabel('Tokens/sec')
    ax7.set_ylabel('Frequency')
    ax7.legend()
    ax7.grid(True, alpha=0.3)
    
    plt.suptitle('LLM Production Monitoring Dashboard', fontsize=16, weight='bold', y=0.995)
    plt.show()
    
    print("\nKey Performance Indicators (KPIs):")
    print("="*70)
    print(f"\n🎯 Availability: {summary['success_rate']:.2f}% (Target: >99.9%)")
    print(f"⚡ P95 Latency: {summary['p95_latency_ms']:.0f}ms (Target: <500ms)")
    print(f"💰 Cost/Request: ${summary['avg_cost_per_request_usd']:.4f}")
    print(f"🚀 Total Tokens: {summary['total_tokens']:,}")
    
    if ratings:
        avg_rating = np.mean(ratings)
        print(f"⭐ User Satisfaction: {avg_rating:.2f}/5.00 (Target: >4.0)")
    
    print(f"\n⚠️ Alerts:")
    if summary['success_rate'] < 99:
        print(f"  • Success rate below 99%: {summary['success_rate']:.1f}%")
    if summary['p95_latency_ms'] > 500:
        print(f"  • P95 latency above 500ms: {summary['p95_latency_ms']:.0f}ms")
    if summary['flagged_requests'] > 0:
        print(f"  • {summary['flagged_requests']} requests flagged")
    if not any([summary['success_rate'] < 99, summary['p95_latency_ms'] > 500, summary['flagged_requests'] > 0]):
        print("  ✅ All systems nominal")

visualize_monitoring_dashboard()

## 2. Distributed Tracing

**Tracing** tracks requests across distributed systems

**Trace Structure**:
```
Request (trace_id: abc123)
├─ API Gateway         [50ms]
├─ Auth Service        [20ms]
├─ LLM Service         [300ms]
│  ├─ Tokenization     [10ms]
│  ├─ Embedding        [30ms]
│  ├─ RAG Retrieval    [80ms]
│  │  ├─ Vector DB     [60ms]
│  │  └─ Reranking     [20ms]
│  ├─ LLM Inference    [150ms]
│  └─ Post-process     [30ms]
└─ Response Format     [10ms]
Total: 380ms
```

**Benefits**:
- Find bottlenecks
- Debug errors
- Optimize critical path
- Understand dependencies

In [ ]:
def simulate_traced_request(logger: LLMLogger, trace_id: str):
    """Simulate a traced LLM request."""
    
    # Root span
    root = logger.start_span(trace_id, "handle_request")
    time.sleep(0.001)
    
    # API Gateway
    gateway = logger.start_span(trace_id, "api_gateway", root.span_id)
    time.sleep(0.05)
    logger.end_span(gateway)
    
    # Auth
    auth = logger.start_span(trace_id, "auth_service", root.span_id)
    time.sleep(0.02)
    logger.end_span(auth)
    
    # LLM Service
    llm_service = logger.start_span(trace_id, "llm_service", root.span_id)
    
    # Tokenization
    tokenize = logger.start_span(trace_id, "tokenization", llm_service.span_id)
    time.sleep(0.01)
    logger.end_span(tokenize)
    
    # Embedding
    embed = logger.start_span(trace_id, "embedding", llm_service.span_id)
    time.sleep(0.03)
    logger.end_span(embed)
    
    # RAG Retrieval
    rag = logger.start_span(trace_id, "rag_retrieval", llm_service.span_id)
    
    vector_db = logger.start_span(trace_id, "vector_db_query", rag.span_id)
    time.sleep(0.06)
    logger.end_span(vector_db)
    
    rerank = logger.start_span(trace_id, "reranking", rag.span_id)
    time.sleep(0.02)
    logger.end_span(rerank)
    
    logger.end_span(rag)
    
    # LLM Inference (longest step)
    inference = logger.start_span(trace_id, "llm_inference", llm_service.span_id)
    time.sleep(0.15)
    logger.end_span(inference)
    
    # Post-processing
    postprocess = logger.start_span(trace_id, "post_processing", llm_service.span_id)
    time.sleep(0.03)
    logger.end_span(postprocess)
    
    logger.end_span(llm_service)
    
    # Response formatting
    response = logger.start_span(trace_id, "response_format", root.span_id)
    time.sleep(0.01)
    logger.end_span(response)
    
    logger.end_span(root)

# Simulate traced request
trace_id = "trace_example"
simulate_traced_request(logger, trace_id)

# Visualize trace
def visualize_trace(logger: LLMLogger, trace_id: str):
    """Visualize distributed trace as waterfall diagram."""
    
    spans = logger.spans[trace_id]
    if not spans:
        print("No spans found for trace")
        return
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Sort by start time
    spans_sorted = sorted(spans, key=lambda s: s.start_time)
    
    # Calculate relative times
    start_time = spans_sorted[0].start_time
    
    colors = {
        'handle_request': 'lightgray',
        'api_gateway': 'lightblue',
        'auth_service': 'lightgreen',
        'llm_service': 'lightyellow',
        'tokenization': 'wheat',
        'embedding': 'peachpuff',
        'rag_retrieval': 'lightcoral',
        'vector_db_query': 'salmon',
        'reranking': 'lightsalmon',
        'llm_inference': 'gold',
        'post_processing': 'khaki',
        'response_format': 'lightcyan'
    }
    
    # Calculate depth (indentation) for each span
    depths = {}
    for span in spans_sorted:
        if span.parent_id is None:
            depths[span.span_id] = 0
        else:
            parent_depth = depths.get(span.parent_id, 0)
            depths[span.span_id] = parent_depth + 1
    
    # Plot spans
    y_pos = 0
    y_labels = []
    y_positions = []
    
    for span in spans_sorted:
        if span.duration_ms is None:
            continue
        
        start_ms = (span.start_time - start_time).total_seconds() * 1000
        duration = span.duration_ms
        depth = depths[span.span_id]
        
        # Plot bar
        color = colors.get(span.name, 'lightgray')
        ax.barh(y_pos, duration, left=start_ms, height=0.8,
               color=color, edgecolor='black', linewidth=1.5)
        
        # Add duration label
        ax.text(start_ms + duration/2, y_pos, f'{duration:.0f}ms',
               ha='center', va='center', fontsize=9, weight='bold')
        
        # Label with indentation
        indent = '  ' * depth
        label = f"{indent}{span.name}"
        y_labels.append(label)
        y_positions.append(y_pos)
        
        y_pos += 1
    
    ax.set_yticks(y_positions)
    ax.set_yticklabels(y_labels, fontsize=10)
    ax.set_xlabel('Time (ms)', fontsize=11, weight='bold')
    ax.set_title(f'Distributed Trace: {trace_id}', fontsize=13, weight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    ax.invert_yaxis()  # Top to bottom
    
    # Total time
    root_span = [s for s in spans if s.parent_id is None][0]
    total_time = root_span.duration_ms
    ax.text(0.98, 0.02, f'Total: {total_time:.0f}ms',
           transform=ax.transAxes, ha='right', va='bottom',
           fontsize=12, weight='bold',
           bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    # Print breakdown
    print("\nTrace Breakdown:")
    print("="*70)
    
    for span in spans_sorted:
        if span.duration_ms and span.parent_id is None:
            continue  # Skip root
        if span.duration_ms:
            depth = depths[span.span_id]
            indent = '  ' * depth
            pct = (span.duration_ms / total_time) * 100
            print(f"{indent}{span.name}: {span.duration_ms:.1f}ms ({pct:.1f}%)")
    
    # Critical path
    print(f"\nTotal Time: {total_time:.1f}ms")
    print("\nBottleneck: llm_inference (150ms - 39% of total)")
    print("Optimization: Use quantization or smaller model")

visualize_trace(logger, trace_id)

## Summary

You've mastered LLM monitoring and observability:
- ✅ LLM-specific logging and metrics
- ✅ Distributed tracing for debugging
- ✅ Cost tracking per request
- ✅ Performance dashboards
- ✅ Error detection and alerting
- ✅ User feedback collection

**Key Metrics to Monitor**:

| Category | Metrics | Target |
|----------|---------|--------|
| **Availability** | Success rate | >99.9% |
| **Latency** | P50, P95, P99 | <500ms P95 |
| **Cost** | $/request, $/1M tokens | Budget dependent |
| **Quality** | User ratings, accuracy | >4.0/5.0 |
| **Safety** | Flagged requests | <0.1% |
| **Throughput** | Requests/sec, tokens/sec | Capacity dependent |

**Monitoring Stack**:
1. **Infrastructure**: Prometheus + Grafana
2. **Application**: DataDog / New Relic
3. **LLM-specific**: LangSmith / Weights & Biases
4. **Tracing**: OpenTelemetry / Jaeger
5. **Logs**: ELK Stack / Splunk

**Alert Conditions**:
- P95 latency > 500ms for 5 minutes
- Error rate > 1% for 2 minutes
- Cost spike > 2x baseline
- Safety violations detected
- GPU utilization < 50% (underutilized)
- Queue depth > 1000 requests

**Best Practices**:
- ✓ Log every request (sample if >1M/day)
- ✓ Track user feedback
- ✓ Monitor costs in real-time
- ✓ Use distributed tracing
- ✓ Set up alerting on key metrics
- ✓ Build dashboards for stakeholders
- ✓ Regular performance reviews
- ✓ A/B test changes

**Next**: Deployment strategies and CI/CD for LLMs